# RT-DETRv2 SportsMOT player detector — val-split benchmark (Kaggle T4)

Pulls [`smallTech/rtdetr-sportsmot`](https://huggingface.co/smallTech/rtdetr-sportsmot)
from the Hub and benchmarks it on the **val split** of
[`Lekim89/sportsmot`](https://huggingface.co/datasets/Lekim89/sportsmot) —
45 annotated sequences the model has **never seen** (training used only the
`train/` split; `test/` has no public ground truth, so it cannot yield
accuracy metrics).

Data comes pre-staged from the `rtdetr-sportsmot-evaluation-prepare-data` CPU kernel
(`evaluation/prepare-data.kaggle.ipynb`), mounted via
`kernel_sources` — never bulk-downloaded from the Hub in-session.

Outputs, all logged and persisted to `benchmarks.json` in the kernel output:
per-sequence and overall COCO mAP/mAR (torchmetrics, same implementation the
trainer reported), per-frame latency percentiles + fps, and environment info.

Run the sibling `smoketest` notebook (a few minutes) before this one.

In [ ]:
# --- 1. Dependencies --------------------------------------------------------
# CRITICAL: do NOT reinstall or upgrade torch OR transformers.
#   * torch: Kaggle's build carries the CUDA kernels for the provisioned GPU;
#     a PyPI wheel breaks with "no kernel image is available".
#   * transformers: the checkpoint must load under the SAME lineage that
#     trained it — an upgraded transformers once loaded this model with the
#     decoder heads randomly re-initialized (mAP 0.04 vs 0.78), and
#     transformers v5 drops rt_detr_v2 recognition entirely.
# Install ONLY torchmetrics (the trainer's mAP implementation, so numbers are
# comparable), under a constraint that freezes both working builds.
import subprocess, sys, tempfile
import torch, transformers

_con = tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False)
_con.write(f"torch=={torch.__version__}\ntransformers=={transformers.__version__}\n")
_con.close()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-c", _con.name, "torchmetrics"],
    check=True,
)
print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
# --- 2. Hugging Face token (optional) ---------------------------------------
# The model is public, so this only raises rate limits. The token rides in the
# private external-secrets dataset (a `secrets` file of KEY=VALUE lines) —
# Kaggle Secrets are dropped on every push, dataset mounts persist. The mount
# layout has changed before, so scan /kaggle/input instead of hard-coding.
import os
from pathlib import Path

def _read_hf_token():
    base = Path("/kaggle/input")
    hits = sorted(base.rglob("secrets")) if base.is_dir() else []
    for hit in hits:
        for line in hit.read_text().splitlines():
            if line.strip().startswith("HF_TOKEN="):
                return line.split("=", 1)[1].strip().strip('"').strip("'")
    return None

_tok = _read_hf_token()
if _tok:
    os.environ["HF_TOKEN"] = _tok
print("HF auth:", "token found" if _tok else "none (public model, lower rate limits)")

In [ ]:
# --- 3. GPU sanity ----------------------------------------------------------
# Fail fast on the wrong accelerator: the API-default P100 (sm_60) has no CUDA
# kernels in Kaggle's torch build; configs pin machine_shape=NvidiaTeslaT4.
import torch

assert torch.cuda.is_available(), "No CUDA GPU — check the kernel's accelerator settings"
cap = torch.cuda.get_device_capability(0)
name = torch.cuda.get_device_name(0)
print(f"GPU: {name} (sm_{cap[0]}{cap[1]})")
assert cap >= (7, 0), f"{name} (sm_{cap[0]}{cap[1]}) unusable: Kaggle torch ships no kernels for it"
x = torch.randn(256, 256, device="cuda") @ torch.randn(256, 256, device="cuda")
torch.cuda.synchronize()
print("CUDA matmul OK:", float(x.sum()))

In [ ]:
# --- 4. Locate + extract the staged val split -------------------------------
# The evaluation/prepare-data CPU kernel staged val/ as sportsmot-val.tar; it is
# mounted via kernel_sources. NOT the .partial.tar — a partial archive would
# silently benchmark on a subset. No Hub-download fallback here: evaluation
# must be reproducible, so a missing mount is an error (run
# evaluation/prepare-data first).
import subprocess
import time

def _find_val_tar():
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    for pattern in ("*/sportsmot-val.tar", "*/*/sportsmot-val.tar",
                    "*/*/*/sportsmot-val.tar"):
        hits = sorted(base.glob(pattern))
        if hits:
            return hits[0]
    return None

VAL_TAR = _find_val_tar()
assert VAL_TAR is not None, (
    "sportsmot-val.tar not mounted — run evaluation/prepare-data "
    "and list its slug in this config's kernel_sources")
# /tmp, NOT /kaggle/working: working becomes the kernel OUTPUT, and an
# extracted ~6 GB tree there bloats it beyond usability (only benchmarks.json
# belongs in the output).
DATA = Path("/tmp/sportsmot-data")
if not (DATA / "val").is_dir():
    print(f"Extracting {VAL_TAR} ...")
    _t = time.time()
    DATA.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "-xf", str(VAL_TAR), "-C", str(DATA)], check=True)
    print(f"extracted in {time.time() - _t:.0f}s")
SEQ_DIRS = sorted(p for p in (DATA / "val").iterdir() if (p / "gt" / "gt.txt").is_file())
print(f"{len(SEQ_DIRS)} annotated val sequences")

In [ ]:
# --- 5. MOTChallenge annotations --------------------------------------------
# gt/gt.txt: frame, track_id, x, y, w, h, conf, class, visibility — same
# parsing as the trainer: conf==0 marks ignore boxes, track ids are irrelevant
# for detection metrics.
from collections import defaultdict

def parse_gt(gt_path):
    """Return {frame_number: [[x, y, w, h], ...]}."""
    frames = defaultdict(list)
    for line in gt_path.read_text().strip().splitlines():
        parts = [p.strip() for p in line.split(",")]
        frame = int(parts[0])
        x, y, w, h = (float(v) for v in parts[2:6])
        conf = int(parts[6])
        if conf == 0 or w <= 0 or h <= 0:
            continue
        frames[frame].append([x, y, w, h])
    return frames

print("gt parser ready")

In [ ]:
# --- 6. Load the fine-tuned model from the Hub ------------------------------
from transformers import AutoImageProcessor, AutoModelForObjectDetection

MODEL_ID = "smallTech/rtdetr-sportsmot"
_t = time.time()
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForObjectDetection.from_pretrained(MODEL_ID).to("cuda").eval()
LOAD_S = time.time() - _t
print(f"loaded {MODEL_ID} in {LOAD_S:.1f}s "
      f"({sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params)")

In [ ]:
# --- 7. Evaluate ------------------------------------------------------------
# Per-frame detection vs gt, per-sequence and overall COCO mAP (torchmetrics,
# same as the trainer: threshold 0.01 so mAP integrates the full PR curve).
# Latency is measured model-side (preprocess + forward + postprocess, CUDA
# synchronized) — the number a deployment would care about.
from PIL import Image
from torchmetrics.detection.mean_ap import MeanAveragePrecision

EVAL_STRIDE = 1                       # full benchmark: every annotated frame
CONF_FOR_MAP = 0.01

overall = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
per_seq = {}
latencies = []
n_frames = n_boxes = 0
t_eval = time.time()

for seq_dir in SEQ_DIRS:
    gt = parse_gt(seq_dir / "gt" / "gt.txt")
    seq_metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    frames = sorted((seq_dir / "img1").glob("*.jpg"))[::EVAL_STRIDE]
    with torch.no_grad():
        for img_path in frames:
            boxes = gt.get(int(img_path.stem), [])
            if not boxes:
                continue
            image = Image.open(img_path).convert("RGB")
            torch.cuda.synchronize(); _t0 = time.perf_counter()
            inputs = processor(images=image, return_tensors="pt").to("cuda")
            outputs = model(**inputs)
            (result,) = processor.post_process_object_detection(
                outputs,
                target_sizes=torch.tensor([image.size[::-1]]).to("cuda"),
                threshold=CONF_FOR_MAP,
            )
            torch.cuda.synchronize()
            latencies.append(time.perf_counter() - _t0)
            pred = [{k: result[k].cpu() for k in ("boxes", "scores", "labels")}]
            gt_xyxy = torch.tensor(
                [[x, y, x + w, y + h] for x, y, w, h in boxes], dtype=torch.float32)
            target = [{"boxes": gt_xyxy,
                       "labels": torch.zeros(len(boxes), dtype=torch.long)}]
            seq_metric.update(pred, target)
            overall.update(pred, target)
            n_frames += 1
            n_boxes += len(boxes)
    m = {k: float(v) for k, v in seq_metric.compute().items() if v.numel() == 1}
    per_seq[seq_dir.name] = m
    print(f"{seq_dir.name}: mAP={m['map']:.4f} mAP50={m['map_50']:.4f} "
          f"({len(frames)} frames)", flush=True)

overall_m = {k: float(v) for k, v in overall.compute().items() if v.numel() == 1}
print(f"\nevaluated {n_frames} frames / {n_boxes} gt boxes "
      f"in {(time.time() - t_eval) / 60:.1f} min")

In [ ]:
import json
# --- 8. Benchmarks: log + persist -------------------------------------------
import statistics

lat_ms = sorted(l * 1000 for l in latencies)
def pct(p):
    return lat_ms[min(len(lat_ms) - 1, int(p / 100 * len(lat_ms)))]

benchmarks = {
    "model": MODEL_ID,
    "dataset": "Lekim89/sportsmot",
    "split": "val (unseen by training; train/ was the only split trained on)",
    "sequences": len(per_seq),
    "frames_evaluated": n_frames,
    "gt_boxes": n_boxes,
    "eval_stride": EVAL_STRIDE,
    "metrics_overall": overall_m,
    "metrics_per_sequence": per_seq,
    "latency_ms": {
        "mean": statistics.mean(lat_ms),
        "p50": pct(50), "p90": pct(90), "p99": pct(99),
        "fps_mean": 1000 / statistics.mean(lat_ms),
    },
    "environment": {
        "gpu": torch.cuda.get_device_name(0),
        "torch": torch.__version__,
        "model_load_s": round(LOAD_S, 1),
    },
}

out = Path("/kaggle/working/benchmarks.json")
out.write_text(json.dumps(benchmarks, indent=2))
print(json.dumps({k: v for k, v in benchmarks.items()
                  if k != "metrics_per_sequence"}, indent=2))
print("=" * 62)
print(f"BENCHMARK COMPLETE — mAP@[.5:.95]={overall_m['map']:.4f}  "
      f"mAP@50={overall_m['map_50']:.4f}  mAP@75={overall_m['map_75']:.4f}")
print(f"mAR@100={overall_m['mar_100']:.4f}  "
      f"latency p50={pct(50):.0f}ms ({1000 / pct(50):.1f} fps on "
      f"{torch.cuda.get_device_name(0)})")
print(f"full per-sequence breakdown: {out} (kernel Output tab)")
print("=" * 62)

In [ ]:
# --- 9. Model-card section publisher -----------------------------------------
# Replaces (or appends) a marker-delimited section in the Hub model card, so
# re-runs update in place instead of stacking duplicates. A legacy section
# with the same header but no markers is replaced up to the next "## ".
import re
from huggingface_hub import HfApi, hf_hub_download

def publish_card_section(marker: str, header: str, body_md: str, commit: str):
    if not os.environ.get("HF_TOKEN"):
        print("no HF_TOKEN — model card NOT updated "
              "(benchmarks.json is in the kernel output)")
        return
    start, end = f"<!-- {marker}:start -->", f"<!-- {marker}:end -->"
    section = f"{start}\n{body_md.strip()}\n{end}"
    card = Path(hf_hub_download(MODEL_ID, "README.md", force_download=True)).read_text()
    if start in card and end in card:
        card = card[:card.index(start)] + section + card[card.index(end) + len(end):]
    elif header in card:                      # legacy: headed section, no markers
        i = card.index(header)
        m = re.search(r"\n## ", card[i + len(header):])
        j = i + len(header) + m.start() + 1 if m else len(card)
        card = card[:i] + section + "\n" + card[j:]
    else:
        card = card.rstrip() + "\n\n" + section + "\n"
    HfApi().upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md",
                        repo_id=MODEL_ID, commit_message=commit)
    print(f"model card updated: {header!r}")

In [ ]:
# --- 10. Publish to the model card -------------------------------------------
o, lat = benchmarks["metrics_overall"], benchmarks["latency_ms"]
rows = "\n".join(
    f"| `{n}` | {m['map']:.3f} | {m['map_50']:.3f} | {m['map_75']:.3f} |"
    for n, m in sorted(benchmarks["metrics_per_sequence"].items(),
                       key=lambda kv: kv[1]["map"], reverse=True))
body = f"""## Benchmark — unseen val split ({benchmarks['sequences']} sequences)

Evaluated on **every annotated frame of the SportsMOT `val` split** —
{benchmarks['frames_evaluated']:,} frames / {benchmarks['gt_boxes']:,} ground-truth boxes across
{benchmarks['sequences']} sequences (basketball, soccer, volleyball) **never used in
training** (training consumed only the `train/` split; `test/` has no public
ground truth). Frame stride {benchmarks['eval_stride']}; same metric implementation as
the training-time validation (torchmetrics COCO mAP, detection threshold 0.01).

| metric | value |
|---|---|
| mAP@[.5:.95] | **{o['map']:.4f}** |
| mAP@50 | {o['map_50']:.4f} |
| mAP@75 | {o['map_75']:.4f} |
| mAR@100 | {o['mar_100']:.4f} |

**Latency** (single image, batch 1, fp32, {benchmarks['environment']['gpu']},
torch {benchmarks['environment']['torch']}): mean {lat['mean']:.1f} ms · p50 {lat['p50']:.1f} ms ·
p90 {lat['p90']:.1f} ms · p99 {lat['p99']:.1f} ms → **{lat['fps_mean']:.1f} fps**. Model load:
{benchmarks['environment']['model_load_s']} s.

<details>
<summary>Per-sequence results (sorted by mAP)</summary>

| sequence | mAP@[.5:.95] | mAP@50 | mAP@75 |
|---|---|---|---|
{rows}

</details>
"""
publish_card_section("benchmark:val", "## Benchmark — unseen val split", body,
                     "Update val benchmark section (evaluation kernel)")